<a href="https://colab.research.google.com/github/kasinadhsarma/Notebooks/blob/main/bilingual_transcribe_telugu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bilingual (original language + English) call transcription

Runs OpenAI Whisper `large-v3` on a free Colab GPU to produce:
1. The **original-language transcript** (Hindi, Telugu, English, or whatever is spoken — Whisper auto-detects it per file, nothing to configure)
2. An **English translation** of the same audio

then pairs them line-by-line into a bilingual `.txt` per file (and a combined file).

Works for mixed batches too — you can upload Hindi calls and Telugu calls together in the same run; each file's language is detected independently.

**Before running:** Runtime menu -> Change runtime type -> select a GPU (T4 is fine, has ~15GB VRAM, plenty for large-v3).

In [1]:
!nvidia-smi

Mon Sep 21 11:19:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Upload your mp3 files

Run this cell and use the file picker to upload all 10 (or however many) mp3 files.

In [3]:
from google.colab import files
import os

os.makedirs("audio", exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    os.rename(name, os.path.join("audio", name))
print("Uploaded:", os.listdir("audio"))

Saving 0f727392-543b-45c9-86ae-78e4c483f446.mp3 to 0f727392-543b-45c9-86ae-78e4c483f446.mp3
Saving 23eed066-fe33-4b33-abda-6fe98b6c73f3.mp3 to 23eed066-fe33-4b33-abda-6fe98b6c73f3.mp3
Saving 62b5570d-406a-4009-93fb-909f0db6d1af.mp3 to 62b5570d-406a-4009-93fb-909f0db6d1af.mp3
Saving 687c0757-0ae6-4785-a750-9293e383c7d2.mp3 to 687c0757-0ae6-4785-a750-9293e383c7d2.mp3
Saving 92968ece-ab04-4c53-bfa9-7b6f463f50a4.mp3 to 92968ece-ab04-4c53-bfa9-7b6f463f50a4.mp3
Saving 741726dc-c6a7-4860-9ae8-bacd68aac8be.mp3 to 741726dc-c6a7-4860-9ae8-bacd68aac8be.mp3
Saving d849bf31-48a4-4157-966d-6e2a97d21261.mp3 to d849bf31-48a4-4157-966d-6e2a97d21261.mp3
Saving dd91724e-7f1f-4629-9df3-9df67e346fd9.mp3 to dd91724e-7f1f-4629-9df3-9df67e346fd9.mp3
Saving f9984491-23f7-4a38-a4e6-4205768dc270.mp3 to f9984491-23f7-4a38-a4e6-4205768dc270.mp3
Saving fe58b5b2-e14a-431d-9f51-55ccfc82b77d.mp3 to fe58b5b2-e14a-431d-9f51-55ccfc82b77d.mp3
Uploaded: ['23eed066-fe33-4b33-abda-6fe98b6c73f3.mp3', 'f9984491-23f7-4a38-a4e6-

### (Alternative) Mount Google Drive instead of uploading

If your mp3s are already in Drive, skip the upload cell above and use this instead — point `AUDIO_DIR` at the Drive folder.

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')
# AUDIO_DIR = "/content/drive/MyDrive/path/to/your/mp3s"
AUDIO_DIR = "audio"

## 2. Load the model (large-v3 on GPU)

In [5]:
import whisper

model = whisper.load_model("large-v3")  # downloads ~3GB once, then cached

100%|██████████████████████████████████████| 2.88G/2.88G [00:23<00:00, 132MiB/s]


## 3. Transcribe (original language) + Translate (English), per file

In [6]:
import glob, json, os

os.makedirs("out", exist_ok=True)
mp3_files = sorted(glob.glob(os.path.join(AUDIO_DIR, "*.mp3")))
print(f"Found {len(mp3_files)} files")

results = {}
for path in mp3_files:
    name = os.path.splitext(os.path.basename(path))[0]

    orig = model.transcribe(path, task="transcribe")   # original language, auto-detected
    en   = model.transcribe(path, task="translate")    # English translation

    lang = orig.get("language", "unknown")
    print(f"Processing: {name}  -> detected language: {lang}")

    results[name] = {"orig": orig, "en": en, "lang": lang}

    lang_dir = os.path.join("out", lang)
    os.makedirs(lang_dir, exist_ok=True)
    with open(os.path.join(lang_dir, f"{name}.orig.json"), "w", encoding="utf-8") as f:
        json.dump(orig, f, ensure_ascii=False, indent=2)
    with open(os.path.join(lang_dir, f"{name}.en.json"), "w", encoding="utf-8") as f:
        json.dump(en, f, ensure_ascii=False, indent=2)

print("Done. Files are grouped under out/<language>/ (e.g. out/te/ for Telugu, out/hi/ for Hindi).")

Found 10 files
Processing: 0f727392-543b-45c9-86ae-78e4c483f446  -> detected language: te
Processing: 23eed066-fe33-4b33-abda-6fe98b6c73f3  -> detected language: te
Processing: 62b5570d-406a-4009-93fb-909f0db6d1af  -> detected language: te
Processing: 687c0757-0ae6-4785-a750-9293e383c7d2  -> detected language: te
Processing: 741726dc-c6a7-4860-9ae8-bacd68aac8be  -> detected language: te
Processing: 92968ece-ab04-4c53-bfa9-7b6f463f50a4  -> detected language: te
Processing: d849bf31-48a4-4157-966d-6e2a97d21261  -> detected language: te
Processing: dd91724e-7f1f-4629-9df3-9df67e346fd9  -> detected language: te
Processing: f9984491-23f7-4a38-a4e6-4205768dc270  -> detected language: te
Processing: fe58b5b2-e14a-431d-9f51-55ccfc82b77d  -> detected language: te
Done. Files are grouped under out/<language>/ (e.g. out/te/ for Telugu, out/hi/ for Hindi).


## 4. Build bilingual line-by-line transcripts

Pairs each original-language segment with the closest English segment by timestamp overlap (segmentation between the two passes can differ slightly, so exact index-pairing isn't reliable).

In [7]:
def best_match(seg, other_segments):
    """Find the other-language segment with the most time overlap."""
    s0, s1 = seg["start"], seg["end"]
    best, best_overlap = None, 0.0
    for o in other_segments:
        overlap = max(0.0, min(s1, o["end"]) - max(s0, o["start"]))
        if overlap > best_overlap:
            best, best_overlap = o, overlap
    return best

os.makedirs("bilingual", exist_ok=True)
combined_by_lang = {}

for name, r in results.items():
    orig_segs = r["orig"]["segments"]
    en_segs = r["en"]["segments"]
    lang = r["lang"]

    lines = [f"=== {name} ({lang}) ===", ""]
    for seg in orig_segs:
        match = best_match(seg, en_segs)
        orig_text = seg["text"].strip()
        en_text = match["text"].strip() if match else ""
        lines.append(f"[{seg['start']:.1f}s] {orig_text}")
        if en_text:
            lines.append(f"    EN: {en_text}")
        lines.append("")

    text = "\n".join(lines)

    lang_dir = os.path.join("bilingual", lang)
    os.makedirs(lang_dir, exist_ok=True)
    with open(os.path.join(lang_dir, f"{name}.bilingual.txt"), "w", encoding="utf-8") as f:
        f.write(text)

    combined_by_lang.setdefault(lang, []).append(text)

for lang, texts in combined_by_lang.items():
    lang_dir = os.path.join("bilingual", lang)
    with open(os.path.join(lang_dir, "combined.bilingual.txt"), "w", encoding="utf-8") as f:
        f.write("\n\n".join(texts))

print("Bilingual transcripts written to ./bilingual/<language>/ (e.g. ./bilingual/te/ for Telugu).")

Bilingual transcripts written to ./bilingual/<language>/ (e.g. ./bilingual/te/ for Telugu).


## 5. Download the results

In [8]:
import shutil
shutil.make_archive("bilingual_transcripts", "zip", "bilingual")

from google.colab import files as gfiles
gfiles.download("bilingual_transcripts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>